# 🚗 Car Image Background Removal — GPU Accelerated
## Image Processing & Computer Vision — Phase 02 (CUDA Edition)

**Institution:** Elsewedy University of Technology  
**Course:** Image Processing | Spring 2026  
**Instructor:** Eng. Hend Adel Ahmed  

---

**Objective:** Remove the background from car images using GrabCut (classical) and U-Net (deep learning), fully optimized to run on **NVIDIA GPU via CUDA** on Windows 11.

**Techniques Covered:**
- ✅ GrabCut (Classical — OpenCV, CPU)
- ✅ U-Net Encoder-Decoder (Deep Learning — PyTorch + CUDA GPU)
- ✅ Mixed Precision Training (FP16) for faster GPU training
- ✅ GPU memory monitoring
- ✅ Comparison & Visualization

## Step 0 — GPU Setup & CUDA Verification

> ⚠️ **Run this cell first.** It confirms your GPU is detected before anything else runs.

In [1]:
import torch
import subprocess
import sys

print("=" * 60)
print("           CUDA / GPU ENVIRONMENT CHECK")
print("=" * 60)

# PyTorch & CUDA versions
print(f"Python        : {sys.version.split()[0]}")
print(f"PyTorch       : {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version  : {torch.version.cuda}")
    print(f"cuDNN Version : {torch.backends.cudnn.version()}")
    print(f"GPU Count     : {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        vram  = props.total_memory / 1024**3
        print(f"  GPU [{i}]      : {props.name}")
        print(f"  VRAM          : {vram:.1f} GB")
        print(f"  Compute Cap.  : {props.major}.{props.minor}")

    # Enable cuDNN autotuner — speeds up fixed-size inputs
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.enabled   = True
    print("\n✅ cuDNN benchmark mode: ON")
    print("✅ GPU is ready for training!")
else:
    print("\n❌ No CUDA GPU detected!")
    print("   Fix: reinstall PyTorch with CUDA:")
    print("   pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121")

print("=" * 60)

           CUDA / GPU ENVIRONMENT CHECK
Python        : 3.13.3
PyTorch       : 2.12.0.dev20260408+cu128
CUDA Available: True
CUDA Version  : 12.8
cuDNN Version : 92000
GPU Count     : 1
  GPU [0]      : NVIDIA GeForce RTX 3050 Ti Laptop GPU
  VRAM          : 4.0 GB
  Compute Cap.  : 8.6

✅ cuDNN benchmark mode: ON
✅ GPU is ready for training!


## Step 1 — Imports & Configuration

In [2]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.amp import GradScaler, autocast         # Mixed precision (FP16)
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF

import random
import time
import copy

os.makedirs('results', exist_ok=True)
os.makedirs('results/transparent', exist_ok=True)

# ── Core config ──────────────────────────────────────────────────────────
DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 256
BATCH    = 8     # increase to 16 if you have ≥8 GB VRAM
EPOCHS   = 20
LR       = 1e-3
SEED     = 42
USE_AMP  = torch.cuda.is_available()   # Mixed precision only on GPU

TRAIN_DIR = 'train'
TEST_DIR  = 'test'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Device         : {DEVICE}")
print(f"PyTorch        : {torch.__version__}")
print(f"OpenCV         : {cv2.__version__}")
print(f"Batch size     : {BATCH}")
print(f"Mixed precision: {USE_AMP}")
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU VRAM       : {vram:.1f} GB")

Device         : cuda
PyTorch        : 2.12.0.dev20260408+cu128
OpenCV         : 4.13.0
Batch size     : 8
Mixed precision: True
GPU VRAM       : 4.0 GB


## Step 2 — GPU Memory Monitor (Helper)

Call `gpu_mem()` any time to see how much VRAM is used.

In [3]:
def gpu_mem(label=""):
    """Print current GPU memory usage. Safe to call even without a GPU."""
    if not torch.cuda.is_available():
        print("[GPU] No CUDA device.")
        return
    alloc     = torch.cuda.memory_allocated(0)  / 1024**2
    reserved  = torch.cuda.memory_reserved(0)   / 1024**2
    total     = torch.cuda.get_device_properties(0).total_memory / 1024**2
    free      = total - reserved
    tag = f"[{label}] " if label else ""
    print(f"GPU {tag}| Allocated: {alloc:.0f} MB  "
          f"Reserved: {reserved:.0f} MB  "
          f"Free: {free:.0f} MB  /  Total: {total:.0f} MB")

gpu_mem("startup")

GPU [startup] | Allocated: 0 MB  Reserved: 0 MB  Free: 4096 MB  /  Total: 4096 MB


## Step 3 — Dataset Loading & Visualization

In [4]:
def collect_image_paths(root):
    paths = []
    for cls in sorted(os.listdir(root)):
        cls_dir = os.path.join(root, cls)
        if not os.path.isdir(cls_dir):
            continue
        for fname in os.listdir(cls_dir):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                paths.append(os.path.join(cls_dir, fname))
    return paths

train_paths = collect_image_paths(TRAIN_DIR)
test_paths  = collect_image_paths(TEST_DIR)

print(f"Train images : {len(train_paths)}")
print(f"Test  images : {len(test_paths)}")

Train images : 3352
Test  images : 813


In [5]:
CLASSES = sorted(os.listdir(TRAIN_DIR))
COLORS  = ['#2196F3','#FF5722','#4CAF50','#9C27B0','#FF9800','#00BCD4','#F44336']

fig, axes = plt.subplots(1, len(CLASSES), figsize=(18, 3))
for ax, cls, col in zip(axes, CLASSES, COLORS):
    cls_dir = os.path.join(TRAIN_DIR, cls)
    sample  = os.path.join(cls_dir, sorted(os.listdir(cls_dir))[0])
    img     = Image.open(sample).convert('RGB')
    ax.imshow(img)
    ax.set_title(cls, fontsize=9, fontweight='bold', color=col)
    ax.axis('off')
plt.suptitle('Sample Images — One Per Class', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('results/01_sample_images.png', dpi=120, bbox_inches='tight')
plt.show()
print("Sample images saved.")

Sample images saved.


## Step 4 — Classical Method: GrabCut (OpenCV, CPU)

GrabCut runs on **CPU** — this is normal and expected. It is fast (~0.5s/image).

In [6]:
def grabcut_remove_bg(img_path, iterations=5, margin=0.10):
    """
    GrabCut background removal (CPU — OpenCV).
    Returns: original (RGB), fg_result (RGB on white bg), mask (binary)
    """
    bgr      = cv2.imread(img_path)
    original = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    h, w     = original.shape[:2]

    mx   = int(margin * w)
    my   = int(margin * h)
    rect = (mx, my, w - 2*mx, h - 2*my)

    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)
    mask_gc   = np.zeros((h, w), np.uint8)

    cv2.grabCut(bgr, mask_gc, rect, bgd_model, fgd_model,
                iterations, cv2.GC_INIT_WITH_RECT)

    fg_mask = np.where(
        (mask_gc == cv2.GC_FGD) | (mask_gc == cv2.GC_PR_FGD), 1, 0
    ).astype(np.uint8)

    kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_OPEN,  kernel, iterations=1)

    fg_result = original.copy()
    fg_result[fg_mask == 0] = 255

    return original, fg_result, fg_mask

print("GrabCut function defined.")

GrabCut function defined.


In [7]:
fig, axes = plt.subplots(len(CLASSES), 3, figsize=(14, len(CLASSES)*3.2))
fig.suptitle('GrabCut Background Removal — One Sample Per Class',
             fontweight='bold', fontsize=13)

for row, cls in enumerate(CLASSES):
    cls_dir  = os.path.join(TRAIN_DIR, cls)
    sample   = os.path.join(cls_dir, sorted(os.listdir(cls_dir))[0])
    original, fg_result, mask = grabcut_remove_bg(sample)

    axes[row, 0].imshow(original)
    axes[row, 0].set_title(f'{cls} — Original', fontsize=9, fontweight='bold')
    axes[row, 0].axis('off')
    axes[row, 1].imshow(mask, cmap='gray')
    axes[row, 1].set_title('Foreground Mask', fontsize=9)
    axes[row, 1].axis('off')
    axes[row, 2].imshow(fg_result)
    axes[row, 2].set_title('Car (BG Removed)', fontsize=9)
    axes[row, 2].axis('off')

plt.tight_layout()
plt.savefig('results/02_grabcut_results.png', dpi=120, bbox_inches='tight')
plt.show()
print("GrabCut results saved.")

GrabCut results saved.


## Step 5 — Deep Learning: U-Net Architecture (GPU)

```
Input (3×256×256)
  → Encoder Block 1 (64)  ──────────────────────────────────┐
  → Encoder Block 2 (128) ──────────────────────────────┐   │
  → Encoder Block 3 (256) ──────────────────────────┐   │   │
  → Encoder Block 4 (512) ──────────────────────┐   │   │   │
  → Bottleneck      (1024)                       │   │   │   │
  → Decoder Block 4 (512) ◄───────────────────── ┘   │   │   │
  → Decoder Block 3 (256) ◄───────────────────────── ┘   │   │
  → Decoder Block 2 (128) ◄───────────────────────────── ┘   │
  → Decoder Block 1 (64)  ◄───────────────────────────────── ┘
  → Output (1×256×256) sigmoid → binary mask
```

In [8]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)


class EncoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = DoubleConv(in_ch, out_ch)
        self.pool = nn.MaxPool2d(2, 2)
    def forward(self, x):
        skip = self.conv(x)
        return self.pool(skip), skip


class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv = DoubleConv(out_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape != skip.shape:
            x = torch.nn.functional.interpolate(
                x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        return self.conv(torch.cat([skip, x], dim=1))


class UNet(nn.Module):
    def __init__(self, in_channels=3, features=[64, 128, 256, 512]):
        super().__init__()
        self.enc1       = EncoderBlock(in_channels, features[0])
        self.enc2       = EncoderBlock(features[0], features[1])
        self.enc3       = EncoderBlock(features[1], features[2])
        self.enc4       = EncoderBlock(features[2], features[3])
        self.bottleneck = DoubleConv(features[3], features[3]*2)
        self.dec4       = DecoderBlock(features[3]*2, features[3], features[3])
        self.dec3       = DecoderBlock(features[3],   features[2], features[2])
        self.dec2       = DecoderBlock(features[2],   features[1], features[1])
        self.dec1       = DecoderBlock(features[1],   features[0], features[0])
        self.out_conv   = nn.Conv2d(features[0], 1, 1)

    def forward(self, x):
        x, s1 = self.enc1(x)
        x, s2 = self.enc2(x)
        x, s3 = self.enc3(x)
        x, s4 = self.enc4(x)
        x = self.bottleneck(x)
        x = self.dec4(x, s4)
        x = self.dec3(x, s3)
        x = self.dec2(x, s2)
        x = self.dec1(x, s1)
        return self.out_conv(x)  # raw logits — sigmoid applied in loss & inference


# ── Instantiate and move to GPU ───────────────────────────────────────────
model = UNet().to(DEVICE)

total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("=" * 55)
print("U-Net Architecture Summary")
print("=" * 55)
print(model)
print()
print(f"Total Parameters     : {total_p:,}")
print(f"Trainable Parameters : {trainable_p:,}")
print(f"Model device         : {next(model.parameters()).device}")
gpu_mem("after model init")

U-Net Architecture Summary
UNet(
  (enc1): EncoderBlock(
    (conv): DoubleConv(
      (block): Sequential(
        (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (5): ReLU(inplace=True)
      )
    )
    (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (enc2): EncoderBlock(
    (conv): DoubleConv(
      (block): Sequential(
        (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): Conv2d(128, 128, 

## Step 6 — Dataset with Auto-Generated GrabCut Masks

In [9]:
class CarBGDataset(Dataset):
    def __init__(self, img_paths, img_size=256, augment=False):
        self.paths     = img_paths
        self.img_size  = img_size
        self.augment   = augment
        self.to_tensor = transforms.ToTensor()
        self.normalize = transforms.Normalize(
            [0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225]
        )

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        try:
            _, _, mask_np = grabcut_remove_bg(path)
            img_np = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        except Exception:
            img_np  = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)
            mask_np = np.zeros((self.img_size, self.img_size),    dtype=np.uint8)

        img_pil  = Image.fromarray(img_np).resize(
            (self.img_size, self.img_size), Image.Resampling.BILINEAR)
        mask_pil = Image.fromarray(mask_np * 255).resize(
            (self.img_size, self.img_size), Image.NEAREST)

        if self.augment and random.random() > 0.5:
            img_pil  = TF.hflip(img_pil)
            mask_pil = TF.hflip(mask_pil)
        if self.augment and random.random() > 0.5:
            angle    = random.uniform(-15, 15)
            img_pil  = TF.rotate(img_pil,  angle)
            mask_pil = TF.rotate(mask_pil, angle)

        img_t  = self.normalize(self.to_tensor(img_pil))
        mask_t = (self.to_tensor(mask_pil) > 0.5).float()

        return img_t, mask_t


# ── Build loaders ─────────────────────────────────────────────────────────
random.shuffle(train_paths)
split    = int(0.85 * len(train_paths))
tr_paths  = train_paths[:split]
val_paths = train_paths[split:]

train_ds = CarBGDataset(tr_paths,   img_size=IMG_SIZE, augment=True)
val_ds   = CarBGDataset(val_paths,  img_size=IMG_SIZE, augment=False)
test_ds  = CarBGDataset(test_paths, img_size=IMG_SIZE, augment=False)

# Windows: num_workers=0 is REQUIRED (no fork support)
# pin_memory=True speeds up CPU→GPU transfers when using CUDA
PIN = torch.cuda.is_available()
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0, pin_memory=PIN)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=PIN)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=PIN)

print(f"Train samples  : {len(train_ds)}")
print(f"Val   samples  : {len(val_ds)}")
print(f"Test  samples  : {len(test_ds)}")
print(f"Batches/epoch  : {len(train_loader)}")
print(f"pin_memory     : {PIN}")

Train samples  : 2849
Val   samples  : 503
Test  samples  : 813
Batches/epoch  : 357
pin_memory     : True


## Step 7 — Loss Function: Dice + BCE

In [10]:
class DiceBCELoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth
        self.bce    = nn.BCEWithLogitsLoss()  # safe with autocast

    def dice_loss(self, pred, target):
        pred   = pred.view(-1)
        target = target.view(-1)
        inter  = (pred * target).sum()
        return 1 - (2.*inter + self.smooth) / \
               (pred.sum() + target.sum() + self.smooth)

    def forward(self, pred, target):
        # BCEWithLogitsLoss expects raw logits (no sigmoid)
        bce = self.bce(pred, target)
        # Dice loss needs probabilities — apply sigmoid here only
        pred_prob = torch.sigmoid(pred)
        return bce + self.dice_loss(pred_prob, target)


def dice_score(pred_bin, target_bin):
    smooth = 1e-6
    inter  = (pred_bin * target_bin).sum()
    return (2.*inter + smooth) / (pred_bin.sum() + target_bin.sum() + smooth)


print("Loss functions defined.")

Loss functions defined.


## Step 8 — Training Loop with Mixed Precision (FP16)

**Mixed Precision (AMP)** uses FP16 for forward/backward passes, keeping FP32 for weight updates.  
This gives **~2× speedup** and **~50% less VRAM** on modern NVIDIA GPUs (Turing / Ampere / Ada).

In [11]:
def train_unet(model, train_loader, val_loader, epochs, lr):
    criterion = DiceBCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3)

    # GradScaler for mixed precision — does nothing if USE_AMP=False
    scaler = GradScaler("cuda", enabled=USE_AMP)

    history   = {'train_loss':[], 'val_loss':[], 'train_dice':[], 'val_dice':[]}
    best_dice = 0.0
    best_wts  = copy.deepcopy(model.state_dict())
    start     = time.time()

    for epoch in range(epochs):
        # ── Train ─────────────────────────────────────────────────────
        model.train()
        tr_loss, tr_dice, n = 0., 0., 0

        for imgs, masks in train_loader:
            # Non-blocking transfers are faster with pin_memory=True
            imgs  = imgs.to(DEVICE, non_blocking=True)
            masks = masks.to(DEVICE, non_blocking=True)

            optimizer.zero_grad()

            # ── Forward with AMP (FP16 on GPU, FP32 on CPU) ───────────
            with autocast(device_type="cuda", enabled=USE_AMP):
                preds = model(imgs)
                loss  = criterion(preds, masks)

            # ── Backward with gradient scaling ────────────────────────
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            tr_loss += loss.item()
            bin_p    = (torch.sigmoid(preds).detach().cpu().numpy() > 0.5).astype(np.uint8)
            bin_m    = masks.cpu().numpy().astype(np.uint8)
            tr_dice += dice_score(bin_p, bin_m)
            n += 1

        # ── Validate ──────────────────────────────────────────────────
        model.eval()
        val_loss, val_dice, nv = 0., 0., 0
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs  = imgs.to(DEVICE, non_blocking=True)
                masks = masks.to(DEVICE, non_blocking=True)
                with autocast(device_type="cuda", enabled=USE_AMP):
                    preds = model(imgs)
                    loss  = criterion(preds, masks)
                val_loss += loss.item()
                bin_p  = (torch.sigmoid(preds).cpu().numpy() > 0.5).astype(np.uint8)
                bin_m  = masks.cpu().numpy().astype(np.uint8)
                val_dice += dice_score(bin_p, bin_m)
                nv += 1

        epoch_vl = val_loss / nv
        epoch_vd = val_dice / nv
        history['train_loss'].append(tr_loss / n)
        history['val_loss'].append(epoch_vl)
        history['train_dice'].append(tr_dice / n)
        history['val_dice'].append(epoch_vd)

        scheduler.step(epoch_vl)

        if epoch_vd > best_dice:
            best_dice = epoch_vd
            best_wts  = copy.deepcopy(model.state_dict())

        amp_tag = "AMP" if USE_AMP else "FP32"
        print(f"Epoch {epoch+1:02d}/{epochs} [{amp_tag}] | "
              f"Train Loss: {tr_loss/n:.4f}  Dice: {tr_dice/n:.4f} | "
              f"Val Loss: {epoch_vl:.4f}  Dice: {epoch_vd:.4f}")

        # Print GPU usage every 5 epochs
        if (epoch + 1) % 5 == 0:
            gpu_mem(f"epoch {epoch+1}")

    elapsed = time.time() - start
    model.load_state_dict(best_wts)
    print(f"\n✅ Best Val Dice: {best_dice:.4f}  |  Training Time: {elapsed:.1f}s")
    return model, history


print("Starting U-Net training on", DEVICE, "...")
gpu_mem("before training")
model, history = train_unet(model, train_loader, val_loader, epochs=EPOCHS, lr=LR)
torch.save(model.state_dict(), 'results/unet_bg_removal_gpu.pth')
gpu_mem("after training")
print("Model saved → results/unet_bg_removal_gpu.pth")

Starting U-Net training on cuda ...
GPU [before training] | Allocated: 120 MB  Reserved: 136 MB  Free: 3960 MB  /  Total: 4096 MB
Epoch 01/20 [AMP] | Train Loss: 0.6313  Dice: 0.7717 | Val Loss: 0.5269  Dice: 0.8107
Epoch 02/20 [AMP] | Train Loss: 0.5181  Dice: 0.8142 | Val Loss: 0.4756  Dice: 0.8299
Epoch 03/20 [AMP] | Train Loss: 0.4730  Dice: 0.8311 | Val Loss: 0.5605  Dice: 0.8264
Epoch 04/20 [AMP] | Train Loss: 0.4502  Dice: 0.8393 | Val Loss: 0.5035  Dice: 0.8143
Epoch 05/20 [AMP] | Train Loss: 0.4358  Dice: 0.8448 | Val Loss: 0.4369  Dice: 0.8488
GPU [epoch 5] | Allocated: 607 MB  Reserved: 2550 MB  Free: 1546 MB  /  Total: 4096 MB
Epoch 06/20 [AMP] | Train Loss: 0.4226  Dice: 0.8501 | Val Loss: 0.3916  Dice: 0.8621
Epoch 07/20 [AMP] | Train Loss: 0.4103  Dice: 0.8541 | Val Loss: 0.3833  Dice: 0.8630
Epoch 08/20 [AMP] | Train Loss: 0.4048  Dice: 0.8570 | Val Loss: 0.4659  Dice: 0.8442
Epoch 09/20 [AMP] | Train Loss: 0.3980  Dice: 0.8591 | Val Loss: 0.3845  Dice: 0.8641
Epoch 10/

## Step 9 — Training Curves

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('U-Net Training Curves — GPU (CUDA)', fontsize=13, fontweight='bold')
eps = range(1, len(history['train_loss'])+1)

axes[0].plot(eps, history['train_loss'], 'b-o', markersize=4, label='Train')
axes[0].plot(eps, history['val_loss'],   'r-o', markersize=4, label='Val')
axes[0].set_title('Dice + BCE Loss', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(eps, history['train_dice'], 'b-o', markersize=4, label='Train')
axes[1].plot(eps, history['val_dice'],   'r-o', markersize=4, label='Val')
axes[1].set_title('Dice Score', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Dice')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results/03_training_curves.png', dpi=130, bbox_inches='tight')
plt.show()
print("Training curves saved.")

Training curves saved.


## Step 10 — Evaluation on Test Set

In [13]:
def evaluate_segmentation(model, loader):
    model.eval()
    all_dice, all_iou = [], []
    with torch.no_grad():
        for imgs, masks in loader:
            imgs = imgs.to(DEVICE, non_blocking=True)
            with autocast(device_type="cuda", enabled=USE_AMP):
                preds = torch.sigmoid(model(imgs))
            bin_p = (preds.cpu().numpy() > 0.5).astype(np.uint8).flatten()
            bin_m = masks.numpy().astype(np.uint8).flatten()

            smooth = 1e-6
            inter  = (bin_p * bin_m).sum()
            dice   = (2*inter + smooth) / (bin_p.sum() + bin_m.sum() + smooth)
            union  = bin_p.sum() + bin_m.sum() - inter
            iou    = (inter + smooth) / (union + smooth)
            all_dice.append(dice)
            all_iou.append(iou)

    mean_dice = np.mean(all_dice)
    mean_iou  = np.mean(all_iou)
    print("=" * 50)
    print("U-Net — Test Set Segmentation Results")
    print("=" * 50)
    print(f"Mean Dice Score : {mean_dice*100:.2f}%")
    print(f"Mean IoU        : {mean_iou*100:.2f}%")
    return mean_dice, mean_iou

unet_dice, unet_iou = evaluate_segmentation(model, test_loader)
gpu_mem("after evaluation")

U-Net — Test Set Segmentation Results
Mean Dice Score : 87.54%
Mean IoU        : 78.13%
GPU [after evaluation] | Allocated: 242 MB  Reserved: 590 MB  Free: 3506 MB  /  Total: 4096 MB


## Step 11 — GrabCut vs U-Net Comparison

In [14]:
def remove_bg_unet(model, img_path, img_size=256):
    img_np   = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    original = img_np.copy()
    img_pil  = Image.fromarray(img_np).resize((img_size, img_size), Image.Resampling.BILINEAR)
    tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])
    inp = tf(img_pil).unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad():
        with autocast(device_type="cuda", enabled=USE_AMP):
            pred = torch.sigmoid(model(inp))[0, 0].cpu().numpy()

    mask      = (pred > 0.5).astype(np.uint8)
    mask_full = cv2.resize(mask, (original.shape[1], original.shape[0]),
                           interpolation=cv2.INTER_NEAREST)
    fg = original.copy()
    fg[mask_full == 0] = 255
    return original, fg, mask_full

print("Inference function defined.")

Inference function defined.


In [15]:
n_samples = min(5, len(test_paths))
samples   = random.sample(test_paths, n_samples)

fig, axes = plt.subplots(n_samples, 5, figsize=(20, n_samples*3.5))
fig.suptitle('Background Removal — GrabCut vs U-Net (GPU)',
             fontweight='bold', fontsize=13)
col_titles = ['Original','GrabCut Mask','GrabCut Result','U-Net Mask','U-Net Result']
for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontweight='bold', fontsize=10)

for row, path in enumerate(samples):
    orig_gc, fg_gc, mask_gc_img = grabcut_remove_bg(path)
    orig_un, fg_un, mask_un     = remove_bg_unet(model, path)
    axes[row,0].imshow(orig_gc)
    axes[row,1].imshow(mask_gc_img, cmap='gray')
    axes[row,2].imshow(fg_gc)
    axes[row,3].imshow(mask_un, cmap='gray')
    axes[row,4].imshow(fg_un)
    for ax in axes[row]: ax.axis('off')

plt.tight_layout()
plt.savefig('results/04_comparison_grabcut_vs_unet.png', dpi=120, bbox_inches='tight')
plt.show()
print("Comparison saved.")

Comparison saved.


## Step 12 — Export: Transparent Background (RGBA PNG)

In [16]:
def save_transparent(img_path, model, save_dir='results/transparent', img_size=256):
    os.makedirs(save_dir, exist_ok=True)
    original, _, mask = remove_bg_unet(model, img_path, img_size)
    rgba    = np.dstack([original, mask * 255])
    out_img = Image.fromarray(rgba.astype(np.uint8), 'RGBA')
    fname   = os.path.splitext(os.path.basename(img_path))[0] + '_nobg.png'
    out_img.save(os.path.join(save_dir, fname))
    return out_img


fig, axes = plt.subplots(1, n_samples, figsize=(16, 4))
fig.suptitle('Exported Cars — Transparent Background (RGBA)',
             fontweight='bold', fontsize=12)

for ax, path in zip(axes, samples):
    rgba_img = save_transparent(path, model)
    block    = 16
    checker  = np.zeros((rgba_img.height, rgba_img.width, 3), dtype=np.uint8)
    for i in range(0, rgba_img.height, block):
        for j in range(0, rgba_img.width, block):
            c = 200 if (i//block + j//block) % 2 == 0 else 255
            checker[i:i+block, j:j+block] = c
    alpha   = np.array(rgba_img)[:,:,3:4] / 255.
    rgb     = np.array(rgba_img)[:,:,:3]
    blended = (rgb*alpha + checker*(1-alpha)).astype(np.uint8)
    ax.imshow(blended)
    ax.set_title(os.path.basename(path)[:20], fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.savefig('results/05_transparent_exports.png', dpi=120, bbox_inches='tight')
plt.show()
print("Transparent exports saved to results/transparent/")

Transparent exports saved to results/transparent/


## Step 13 — Quantitative Results Summary

In [17]:
gc_dices = []
for path in test_paths[:100]:
    try:
        _, _, mask_gc_q = grabcut_remove_bg(path)
        _, _, mask_un_q = remove_bg_unet(model, path)
        mask_gc_r = cv2.resize(mask_gc_q,
                               (mask_un_q.shape[1], mask_un_q.shape[0]),
                               interpolation=cv2.INTER_NEAREST)
        smooth = 1e-6
        inter  = (mask_gc_r * mask_un_q).sum()
        d      = (2*inter+smooth)/(mask_gc_r.sum()+mask_un_q.sum()+smooth)
        gc_dices.append(d)
    except Exception:
        pass

gc_mean_dice = np.mean(gc_dices)

print("=" * 65)
print(f"{'Method':<25} {'Dice Score':>15} {'IoU':>15}")
print("=" * 65)
print(f"{'GrabCut (Classical)':<25} {gc_mean_dice*100:>14.2f}%  {'N/A':>14}")
print(f"{'U-Net (GPU + AMP)':<25} {unet_dice*100:>14.2f}%  {unet_iou*100:>13.2f}%")
print("=" * 65)

fig, ax = plt.subplots(figsize=(8, 5))
methods = ['GrabCut\n(Classical)', f'U-Net\n(GPU {"AMP" if USE_AMP else "FP32"})']
dices   = [gc_mean_dice*100, unet_dice*100]
colors  = ['#FF9800', '#2196F3']
bars    = ax.bar(methods, dices, color=colors, edgecolor='black', alpha=0.9, width=0.4)
for bar, val in zip(bars, dices):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{val:.2f}%', ha='center', fontsize=12, fontweight='bold')
ax.set_ylabel('Dice Score (%)', fontsize=12)
ax.set_title('GrabCut vs U-Net — Dice Score Comparison (GPU Edition)',
             fontweight='bold', fontsize=13)
ax.set_ylim(0, 110)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('results/06_method_comparison.png', dpi=130, bbox_inches='tight')
plt.show()
gpu_mem("final")

Method                         Dice Score             IoU
GrabCut (Classical)                86.54%             N/A
U-Net (GPU + AMP)                  87.54%          78.13%
GPU [final] | Allocated: 242 MB  Reserved: 590 MB  Free: 3506 MB  /  Total: 4096 MB


## Conclusion

| Component | GrabCut | U-Net (GPU) |
|---|---|---|
| Type | Classical (OpenCV) | Deep Learning (PyTorch) |
| Hardware | CPU | CUDA GPU |
| Precision | FP64 | FP16 (Mixed AMP) |
| Supervision | Unsupervised | Self-supervised (GrabCut masks) |
| Speed | ~0.5s/img | GPU batch inference |
| Quality | Good for simple bg | Better at complex edges |
| Windows support | ✅ Full | ✅ `num_workers=0` required |

**GPU Optimizations Applied:**
- ✅ `torch.cuda.amp.autocast` — Mixed precision FP16 forward pass
- ✅ `GradScaler` — Prevents FP16 underflow during backward pass
- ✅ `cudnn.benchmark = True` — Autotuner for fixed input sizes
- ✅ `pin_memory=True` — Faster CPU→GPU data transfers
- ✅ `non_blocking=True` — Async data transfers overlap with compute
- ✅ `num_workers=0` — Windows-safe DataLoader
- ✅ `gpu_mem()` helper — Live VRAM monitoring throughout training